# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Artasam/Machine-Learning/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Ranking / scoring, built on top of a binary classification sub-task.**

Lane 2's real deliverable is a ranked review queue -- "which pages should a reviewer look at
first" -- so on the standard task-type mapping ("Which ones first?") that is fundamentally a
**ranking/scoring** problem: target = a priority score, metric = precision@K.

But the queue isn't built by ranking directly. The starter pipeline (and my lane) first frames
a **classification** sub-task -- "will this page's label be declining, yes or no" -- trains a
model to output a *probability* of decline, then blends that probability with the transparent
baseline score into the final ranking. So the honest framing is two layers: classify first,
rank second. That also matches why the metric has to be a ranking metric (precision@K), not a
plain classification metric like accuracy -- see Section 3.

The check below is why a classification sub-task is even reasonable here: my label is binary,
so I need to know its balance before assuming "declining vs not" is a workable split rather
than a rare-event problem in disguise.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down")

counts = df["is_declining_label"].value_counts()
rate = df["is_declining_label"].mean() * 100

print("is_declining_label counts:")
print(counts)
print(f"\nBase rate of 'declining': {rate:.1f}% of {len(df):,} pages")
print("Roughly balanced (not a 2%-positive rare-event problem), so an ordinary binary")
print("classifier -- not special imbalance handling -- is a reasonable starting sub-task.")


is_declining_label counts:
is_declining_label
True     16262
False    13738
Name: count, dtype: int64

Base rate of 'declining': 54.2% of 30,000 pages
Roughly balanced (not a 2%-positive rare-event problem), so an ordinary binary
classifier -- not special imbalance handling -- is a reasonable starting sub-task.


## 2. Target or proxy

**Target: `is_declining_label = (trend_direction == "down")` -- and I'm naming it a proxy
label, not an observed outcome.**

`trend_direction` is not a raw measurement; it's a **defined rule** the starter pipeline
computes from `trend_pct`, which itself compares `impressions_last_30d` to
`impressions_prev_30d` inside the *same* 90-day export window (`down` when that percentage
change is below -20%). Nothing about it looks forward in time -- it describes what already
happened up to the moment the CSV was built, not what will happen next.

That matters because of the rule from the framing skill: *the target must be observed, not
defined, or the model just learns the rule instead of the world.* Right now I'm using the
starter's defined-rule label on purpose, because it's what the reference pipeline (and its
published Precision@50 numbers) is built on, and I want a real baseline to compare against
before I redesign anything. But I'm flagging honestly, same as I did in Week 1, that a stronger
version of this project would replace it with a genuinely observed future outcome (e.g. prior
90 days of features -> decline over the *next* 30 days), which is a Week 3+ leakage-audit task,
not something to fix today.

The check below shows the rule in numbers: how `trend_direction` buckets map onto the
underlying `trend_pct` values, so "defined rule" isn't just an assertion -- I can see the
threshold with my own eyes.

In [2]:
print("trend_direction buckets (all 5, not just my binary label):")
print(df["trend_direction"].value_counts())
print()
print("trend_pct summary (the number trend_direction is thresholded from):")
print(df["trend_pct"].describe().round(1))
print()
print("This confirms 'down' is a THRESHOLD on trend_pct (< -20%), computed from the SAME")
print("90-day window as the export -- not a future outcome. It is a proxy label I am")
print("adopting deliberately for now, not one I'm claiming is ideal.")


trend_direction buckets (all 5, not just my binary label):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

trend_pct summary (the number trend_direction is thresholded from):
count    26612.0
mean        -4.8
std        473.9
min       -100.0
25%        -62.6
50%        -33.5
75%          0.0
max      44900.0
Name: trend_pct, dtype: float64

This confirms 'down' is a THRESHOLD on trend_pct (< -20%), computed from the SAME
90-day window as the export -- not a future outcome. It is a proxy label I am
adopting deliberately for now, not one I'm claiming is ideal.


## 3. Success metric

**Metric: Precision@50.**

This matches the real decision from my Week 1 framing: a content team can only act on a fixed
weekly capacity, and I set that capacity at 50 reviewed pages/week. Precision@50 asks exactly
the question that matters operationally -- "of the top 50 pages the queue says to review first,
how many are actually genuinely worth reviewing?" -- rather than a generic metric like accuracy,
which would reward the model for getting the other 29,950 *uninteresting* pages right too,
something nobody asked for.

For reference (not reproduced here, since it comes from actually running `scripts/run_all.py`
and is already verified in this repo's `outputs/model_report.md`): the baseline rule scores
Precision@50 = 0.240, and the random forest scores 0.740. I'm not re-deriving that number in
this notebook -- Week 5 (`w05_model`) is where I train and evaluate my own model. What I *can*
do here, honestly, is show the metric's mechanics on a toy sort, so I understand what the
number means before I chase it.

**One thing this check taught me that I want to be upfront about:** my proxy label's base rate
is 54.2% (from Section 1) -- meaning over half of ALL pages count as "declining." That's high
enough that almost *any* top-50 slice, even a bad one, will show a deceptively decent-looking
Precision@50 purely by chance. So the number alone isn't enough -- it has to be read against
the 54.2% base-rate benchmark, and ultimately against the reference pipeline's client-holdout
Precision@50, not against a random slice of the full data the way I demonstrate below.

One thing the toy sort already reveals: sorting by `impressions_90d` alone scores *below* the 
54.2% base rate (0.420 vs 0.542), which is an early signal that high-impression pages are 
anti-correlated with decline — they tend to be big, stable, established pages. Any real model 
or ranking formula that naively boosts high-impression pages will therefore hurt precision, 
not help it.

In [3]:
import numpy as np

def precision_at_k(df_sorted, label_col, k=50):
    return df_sorted.head(k)[label_col].mean()

base_rate = df["is_declining_label"].mean()

# Toy sort 1: purely random order (no signal at all)
random_sorted = df.sample(frac=1, random_state=42)
p50_random = precision_at_k(random_sorted, "is_declining_label")

# Toy sort 2: a naive single-signal sort (NOT the real baseline formula -- just to show
# that even a plausible-looking rule can underperform once precision@K is actually measured)
naive_sorted = df.sort_values("impressions_90d", ascending=False)
p50_naive = precision_at_k(naive_sorted, "is_declining_label")

print(f"Base rate (any 50 pages, on average, would score close to this): {base_rate:.3f}")
print(f"Precision@50, random order (this run):                          {p50_random:.3f}")
print(f"Precision@50, naive sort by impressions_90d only:                {p50_naive:.3f}")
print()
print("Neither toy sort is my real baseline or model -- both are here only to show that")
print("Precision@50 is noisy at k=50 and must be judged against the base rate, not against")
print("zero. The real comparison (baseline formula vs. trained model, on held-out clients)")
print("is Week 4 (ML-07) and Week 5 (ML-08) work, not this notebook's job.")


Base rate (any 50 pages, on average, would score close to this): 0.542
Precision@50, random order (this run):                          0.620
Precision@50, naive sort by impressions_90d only:                0.420

Neither toy sort is my real baseline or model -- both are here only to show that
Precision@50 is noisy at k=50 and must be judged against the base rate, not against
zero. The real comparison (baseline formula vs. trained model, on held-out clients)
is Week 4 (ML-07) and Week 5 (ML-08) work, not this notebook's job.


## 4. The unit of analysis, as a real dataframe

**One row = one page (`content_id`), nested inside one client (`client_id`).**

Every row in the starter file already respects this grain -- `content_id` is unique per row,
and each page belongs to exactly one client. That's the same unit of analysis I stated in
Week 1, and it's the grain the classification sub-task predicts on (one probability per page)
and the ranking sub-task ranks on (one queue position per page, per client's own capacity).

Note: `word_count` is blank for 7,699 rows in this slice — the data dictionary flags this as 
systematic by content type (e.g. `feedly article` rows have no keyword or word-count data at 
all), not random missingness. A blind `fillna(0)` would silently encode content type into the 
feature. The right fix (adding a `has_word_count` flag alongside imputation) is a Week 3 data 
contract task, not today's — but the NaN in the sample above is intentional, not a loading error.


In [4]:
lane_cols = ["content_id", "client_id", "impressions_90d", "ctr", "avg_position",
             "days_since_last_update", "word_count", "engagement_rate", "scroll_rate",
             "trend_direction"]

print(df[lane_cols].head(5).to_string())
print()
print(f"Rows: {len(df):,}  |  Unique content_id: {df['content_id'].nunique():,}  "
      f"(matches row count -> one row per page, confirmed)")
print(f"Unique client_id: {df['client_id'].nunique()}  |  "
      f"Avg pages per client: {len(df) / df['client_id'].nunique():.1f}")


             content_id          client_id  impressions_90d   ctr  avg_position  days_since_last_update  word_count  engagement_rate  scroll_rate trend_direction
0  content_304f48230142  client_f369cb89fc             3803  0.76          10.6                      20      3221.0             5.88         4.55            down
1  content_a1fb4e703a9e  client_4e07408562            15320  0.05          20.3                      25      2481.0             0.00        10.00            down
2  content_9aa793d4d895  client_7f2253d7e2            12581  0.09          36.5                      20      3515.0             0.00        28.57            down
3  content_331d6c4de07b  client_19581e27de            11751  0.49           6.2                      22         NaN             1.28         3.45          stable
4  content_d99b7a2d90ca  client_3fdba35f04            19140  0.13          44.0                      14      2803.0             0.00        24.29            down

Rows: 30,000  |  Unique con

## 5. Why ML beats a fixed rule here

The starter's own baseline (`02_baseline_score.py`) already IS a fixed rule -- a hand-weighted
formula (`0.40 * visibility + 0.30 * freshness_risk + 0.25 * position_opportunity +
0.05 * depth_gap`). Those weights are a guess, fixed for every page, everywhere. The problem is
that the signals behind them don't move together in one predictable direction -- they interact,
and a single linear formula can't bend to capture that.

The check below shows this directly: I cross `position_tier` against `word_count_tier` and
look at the decline rate in each cell. If the world were simple, decline rate would move
smoothly in one direction as position improves or word count grows. It doesn't -- `top_3`
pages have an unusually *low* decline rate at almost every word-count tier, while `page_3_5`
and `striking` pages (worse positions) have a *higher* rate than `deep` pages (worse still).
That's a real, non-monotonic interaction: position and word count don't independently add up
to a decline score, they combine in page-specific ways a fixed linear weight can't represent
without either overfitting the rule by hand or missing real cases.

The near-zero correlations underneath confirm the individual signals aren't redundant with each
other either -- each carries separate information, which is exactly the "many signals, tangled"
condition where the framing skill says a plain if-statement stops being enough, and where the
reference pipeline's own ~3x precision lift (0.240 -> 0.740) is evidence a learned model
captures interactions the fixed formula leaves on the table.

The linear correlation between position and CTR appears weak (-0.08) even after filtering 
out the no-data zeros -- this is because the true relationship is non-linear (exponential 
drop in clicks as position worsens), so Pearson correlation understates the real dependency. 
The two signals do interact, but not in a simple straight-line way -- which reinforces, rather 
than contradicts, the argument that a learned model handles them better than a hand-written formula.

In [5]:
pivot = df.pivot_table(index="position_tier", columns="word_count_tier",
                        values="is_declining_label", aggfunc="mean").round(2)
print("Decline rate by position_tier x word_count_tier (non-monotonic -> real interaction):")
print(pivot)
print()

corr_1 = df["impressions_90d"].corr(df["days_since_last_update"])
valid_pos = df["avg_position"] > 0
corr_2 = df.loc[valid_pos, "avg_position"].corr(df.loc[valid_pos, "ctr"])

print(f"corr(impressions_90d, days_since_last_update): {corr_1:.3f}  (near zero -> independent signal)")
print(f"corr(avg_position, ctr), position>0 only:       {corr_2:.3f}  (linear corr weak -- relationship is non-linear/log-scale, not truly independent)")
print()
print("Weak pairwise correlations + a non-monotonic interaction table together are the")
print("evidence: the pattern is real but too tangled for one hand-written weighted formula.")


Decline rate by position_tier x word_count_tier (non-monotonic -> real interaction):
word_count_tier  1000-2000  2000-3500  3500+  <1000
position_tier                                      
deep                  0.39       0.49   0.52   0.29
page_1                0.59       0.58   0.61   0.30
page_3_5              0.70       0.62   0.67   0.33
striking              0.70       0.61   0.65   0.33
top_3                 0.11       0.47   0.09   0.08

corr(impressions_90d, days_since_last_update): 0.082  (near zero -> independent signal)
corr(avg_position, ctr), position>0 only:       -0.080  (linear corr weak -- relationship is non-linear/log-scale, not truly independent)

Weak pairwise correlations + a non-monotonic interaction table together are the
evidence: the pattern is real but too tangled for one hand-written weighted formula.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled -- markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere (all IDs are the repo's pseudonymized hashes)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` -- then submit your repo URL on the card. Done.
